<a id="top"></a>

# Lab L0806: write the description, design the schema

```yaml
title: "Lab L0806: write the description, design the schema"
keywords: tool description, rich description, parameter schema, typed function, Literal enum, Annotated, per-field description, required, bind_tools, validate arguments, lab
estimated duration: 30
```

> **Lesson:** L08. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L08/objectives.md). Solutions: `L0806_lab_solutions.ipynb`. Pairs with the [L0805 demo](L0805_lecture.ipynb).
>
> **Pure Python — no API key needed.** L08 is about *design judgment*, and you can practice that offline. In this course a tool **is** a typed Python function: `bind_tools` infers the JSON definition (name, description, parameter schema) from it. So "designing the schema" means designing the function's **signature** (types, required-ness, `Literal` enums) and its **docstring** — and then inspecting the schema LangChain infers.
>
> **After this lab you can:** write a description *for the model*, design a tight tool by typing the function well (`Annotated` descriptions, a `Literal` enum, required params), and validate sample arguments against the inferred schema — all in pure Python.

## Contents

- [1. Setup — a sparse tool to improve](#1-setup--a-sparse-tool-to-improve)
- [2. Problem 1 — Rewrite the description for the model](#2-problem-1--rewrite-the-description-for-the-model)
- [3. Problem 2 — Type the function (and read its inferred schema)](#3-problem-2--type-the-function-and-read-its-inferred-schema)
- [4. Problem 3 — Validate sample arguments against your schema](#4-problem-3--validate-sample-arguments-against-your-schema)
- [5. Problem 5 — Why an enum over a free string? (written)](#5-problem-5--why-an-enum-over-a-free-string-written)
- [6. Self-check](#6-self-check)

## 1. Setup — a sparse tool to improve

A `set_priority(ticket_id, level)` tool with a **sparse** docstring and a **loose** signature (`level` is any string, and even optional). Because a tool *is* a typed function here, improving the tool means improving the function. Your job across the lab is to make it a tool a model can use correctly on first read.

In [ ]:
import json
from typing import Annotated, Literal

from langchain_core.utils.function_calling import convert_to_openai_tool


# A SPARSE tool: a bare docstring and a loose signature -- `level` is any string and
# even optional. In this course a tool IS a typed function; bind_tools infers the
# JSON definition printed below from it. So "improve the schema" == "improve the
# function's signature and docstring."
def set_priority(ticket_id: str, level: str = "low") -> str:
    """Sets priority."""  # sparse: no when / what / return guidance
    return f"{ticket_id} -> {level}"


def inferred_schema(fn: object) -> dict[str, object]:
    """The JSON tool definition LangChain infers from a function (name, description, params)."""
    return convert_to_openai_tool(fn)["function"]


print("sparse definition bind_tools would send:")
print(json.dumps(inferred_schema(set_priority), indent=2))

[↑ Back to top](#top)

## 2. Problem 1 — Rewrite the description for the model

Write `RICH_DESCRIPTION`: a 3–5 sentence string that answers all four questions from the lecture — *what it does*, *when to call it*, *when NOT to call it*, *what it returns*. Include the accepted `level` values and a one-line example. This becomes the function's **docstring** in Problem 2 — the description the model actually sees. The check below asserts your description mentions the key facts; the *wording* is yours.

In [ ]:
RICH_DESCRIPTION = ""  # TODO: 3-5 sentences, written for the model's selection step

# Lightweight check that the description covers the essentials (not graded on wording):
low = RICH_DESCRIPTION.lower()
for needle in ["ticket", "low", "high", "return"]:
    print(f"mentions {needle!r}:", needle in low)
assert len(RICH_DESCRIPTION.split()) >= 25, "aim for 3-5 real sentences"

[↑ Back to top](#top)

## 3. Problem 2 — Type the function (and read its inferred schema)

Design the tight tool by **typing the function**: write `set_priority` with `ticket_id` and `level` both **required** (no defaults); `level` a `Literal["low", "medium", "high"]` (that becomes an **enum**); and each parameter `Annotated[type, "..."]` with a per-field description carrying an example or a constraint. Reuse your `RICH_DESCRIPTION` as the docstring. Then read the schema LangChain infers — `TIGHT_SCHEMA` is exactly the parameter block `bind_tools` would send.

In [ ]:
# TODO: write set_priority(ticket_id, level) as a typed function:
#   - ticket_id: Annotated[str, "...an example..."]         (required -- no default)
#   - level:     Annotated[Literal["low","medium","high"], "..."]   (required)
# Give it a one-line docstring, then set its real description:
#   set_priority.__doc__ = RICH_DESCRIPTION
# Finally replace the placeholder below with inferred_schema(set_priority)["parameters"].
TIGHT_SCHEMA: dict[str, object] = {}  # replace with inferred_schema(set_priority)["parameters"]
raise NotImplementedError("build the typed set_priority, then TIGHT_SCHEMA")

[↑ Back to top](#top)

## 4. Problem 3 — Validate sample arguments against your schema

Implement `validate(args, schema)` (a small subset of JSON-Schema validation): every `required` key present; each present value matches its property `type` (`string`->`str`, `integer`->`int`); and if a property has an `enum`, the value is in it. Return `"ok"` or a short reason string. Run it over the sample calls.

In [ ]:
SAMPLES = [
    {"ticket_id": "T-42", "level": "high"},  # ok
    {"ticket_id": "T-7"},  # missing required level
    {"ticket_id": "T-9", "level": "urgent"},  # not in enum
    {"ticket_id": 9, "level": "low"},  # wrong type for ticket_id
]


TYPES = {"string": str, "integer": int, "boolean": bool}


def validate(args: dict[str, object], schema: dict[str, object]) -> str:
    # TODO: check required keys present; check each present value's type;
    #       check enum membership where the property defines an enum.
    raise NotImplementedError("your code here")


for s in SAMPLES:
    print(s, "->", validate(s, TIGHT_SCHEMA))

[↑ Back to top](#top)

## 5. Problem 5 — Why an enum over a free string? (written)

Your tight schema makes `level` an enum instead of a free-form string. In a sentence or two: what does the enum buy you in terms of *model behavior* and *validation* (think about the L0805 demo's sparse-vs-rich contrast)?

_Write your answer by editing this cell (double-click)._

_TODO_

[↑ Back to top](#top)

## 6. Self-check

Compare against `L0806_lab_solutions.ipynb`. Done when your `RICH_DESCRIPTION` answers all four questions, your typed `set_priority` makes both fields required with a `Literal` enum on `level` (so `TIGHT_SCHEMA` shows `required` + `enum` + per-field descriptions), and `validate` returns `ok` for the first sample and a useful reason for the other three.

[↑ Back to top](#top)